<div style="line-height: 1.8;">

# **Genome Annotation v1.0 — Bakta-primary release**

На заключительном этапе сформировали выпуск `Genome Annotation v1.0` в каталоге `results/release_v1_bakta/`. В основу включили структурную аннотацию Bakta и функциональный каталог 4315 CDS, дополненный сведениями eggNOG и InterPro. Предсказанные области PHASTEST и системы DefenseFinder представили отдельными таблицами и геномными треками.

Референс NCBI использовали для оценки соответствий и расхождений, без переноса дополнительных CDS в основной набор. Самостоятельность аннотации определяли как отдельный запуск по FASTA; независимость от сведений известных функциональных баз при этом не предполагали.

</div>

<div style="line-height: 1.8;">

## **1. Входные данные выпуска**

В состав входных данных включили исходный FASTA, файлы структурной аннотации Bakta, объединённую функциональную таблицу и результаты специализированных анализов. Проверили наличие обязательных файлов и зафиксировали их расположение.

Сведения Bakta, eggNOG и InterPro ранее объединили в `ecoli_complete_annotation.tsv`. При сборке выпуска использовали эту таблицу без повторного объединения функциональных источников. Сохранённый GenBank NCBI включили только в этап референсной проверки.

</div>

In [1]:
from pathlib import Path
import json, hashlib, shutil, os
from urllib.parse import quote
import numpy as np
import pandas as pd
from Bio import SeqIO
from IPython.display import display

import os

configured_root = os.environ.get("GENOME_ANNOTATION_PROJECT_ROOT")
if configured_root:
    project_dir = Path(configured_root).expanduser().resolve()
else:
    candidates = [candidate for base in (Path.cwd(), *Path.cwd().parents)
                  for candidate in (base, base / "projects/genome_annotation")]
    project_dir = next((p for p in candidates if (p / "data/raw/genome.fna").is_file()), None)
if project_dir is None or not (project_dir / "data/raw/genome.fna").is_file():
    raise FileNotFoundError("Project root not found")
PROJECT_ROOT = project_dir
INPUTS = {
    "genome_fna": PROJECT_ROOT / "data/raw/genome.fna",
    "bakta_gbff": PROJECT_ROOT / "results/structural/bakta/ecoli.gbff",
    "bakta_gff3": PROJECT_ROOT / "results/structural/bakta/ecoli.gff3",
    "bakta_faa": PROJECT_ROOT / "results/structural/bakta/ecoli.faa",
    "functional_table": PROJECT_ROOT / "results/integrated/ecoli_complete_annotation.tsv",
    "phastest_regions": PROJECT_ROOT / "results/specialized/phastest/phastest_regions_annotated.tsv",
    "defense_systems": PROJECT_ROOT / "results/specialized/defensefinder/defense_systems_with_coordinates.tsv",
    "defense_genes": PROJECT_ROOT / "results/specialized/defensefinder/defense_genes_with_coordinates.tsv",
    "defense_faa": PROJECT_ROOT / "results/specialized/defensefinder/defensefinder_component_proteins.faa",
}
NCBI_GBK = PROJECT_ROOT / "data/processed/NC_000913.3.gbk"

missing = [name for name, path in INPUTS.items() if not path.is_file()]
display(pd.DataFrame([{"name":k, "exists": p.is_file(), "path":str(p.relative_to(PROJECT_ROOT))} for k,p in INPUTS.items()]))
if missing:
    raise FileNotFoundError("Missing inputs: " + ", ".join(missing))

RELEASE = PROJECT_ROOT / "results/release_v1_bakta"
TABLES, QC, TRACKS = (RELEASE / p for p in ("tables", "qc", "tracks"))
for folder in (TABLES, QC, TRACKS):
    folder.mkdir(parents=True, exist_ok=True)

,name,exists,path
0,genome_fna,True,data/raw/genome.fna
1,bakta_gbff,True,results/structural/bakta/ecoli.gbff
2,bakta_gff3,True,results/structural/bakta/ecoli.gff3
3,bakta_faa,True,results/structural/bakta/ecoli.faa
4,functional_table,True,results/integrated/ecoli_complete_annotation.tsv
5,phastest_regions,True,results/specialized/phastest/phastest_regions_...
6,defense_systems,True,results/specialized/defensefinder/defense_syst...
7,defense_genes,True,results/specialized/defensefinder/defense_gene...
8,defense_faa,True,results/specialized/defensefinder/defensefinde...


<div style="line-height: 1.8;">

## **2. Первичная последовательность и аннотация Bakta**

Подтвердили побуквенное совпадение последовательности из GenBank Bakta с исходным FASTA. Из аннотации извлекли все 9378 записей геномных признаков, включая `source`, `gene`, CDS и РНК. Это число характеризовало объём записей, а не количество независимых генов.

Для дополнительных GFF3-треков использовали идентификатор хромосомы из исходного GFF3 Bakta. Координаты в сводных таблицах представили с отсчётом от 1 и включением обеих границ; дополнительно сохранили исходное представление расположения признаков в Biopython.

</div>

In [2]:
input_genome = SeqIO.read(INPUTS["genome_fna"], "fasta")
bakta_genome = SeqIO.read(INPUTS["bakta_gbff"], "genbank")
assert len(input_genome.seq) == len(bakta_genome.seq), "Bakta sequence length mismatch"
assert str(input_genome.seq).upper() == str(bakta_genome.seq).upper(), (
    "Bakta sequence mismatch"
)
GENOME_LENGTH = len(input_genome.seq)
print("Genome length:", GENOME_LENGTH, "bp")

seqids = set()
with open(INPUTS["bakta_gff3"], encoding="utf-8") as handle:
    for line in handle:
        if line.startswith("##FASTA"):
            break
        if line.startswith("#") or not line.strip():
            continue
        pieces = line.rstrip("\n").split("\t")
        if len(pieces) == 9:
            seqids.add(pieces[0])
assert len(seqids) == 1, f"Unexpected GFF sequence IDs: {seqids}"
SEQID = next(iter(seqids))
print("Sequence ID:", SEQID)

def qual_first(f, name):
    vals = f.qualifiers.get(name, [])
    return str(vals[0]) if vals else None

features = []
for i, feat in enumerate(bakta_genome.features):
    if feat.location is None:
        continue
    parts = list(feat.location.parts)
    features.append({
        "feature_uid": f"BAKTA_{i:06d}",
        "feature_type": feat.type,
        "locus_tag": qual_first(feat, "locus_tag"),
        "gene": qual_first(feat, "gene"),
        "product": qual_first(feat, "product"),
        "start": int(feat.location.start) + 1,
        "end": int(feat.location.end),
        "strand": feat.location.strand,
        "is_compound": len(parts) != 1,
        "is_pseudogene": bool("pseudo" in feat.qualifiers or "pseudogene" in feat.qualifiers),
        "location_genbank": str(feat.location),
        "qualifiers_json": json.dumps(feat.qualifiers, ensure_ascii=False),
    })
bakta_features = pd.DataFrame(features)
assert bakta_features.start.ge(1).all()
assert bakta_features.end.le(GENOME_LENGTH).all()
assert bakta_features.end.ge(bakta_features.start).all()
bakta_features.to_csv(TABLES / "bakta_all_features.tsv", sep="\t", index=False)
display(bakta_features.groupby("feature_type").size().sort_values(ascending=False).to_frame("count"))

Genome length: 4641652 bp
Sequence ID: contig_1


,count
feature_type,
gene,4658
CDS,4315
ncRNA,232
tRNA,88
regulatory,57
rRNA,22
rep_origin,2
repeat_region,2
source,1


<div style="line-height: 1.8;">

## **3. Основной каталог CDS Bakta + eggNOG + InterPro**

В готовой функциональной таблице проверили уникальность идентификаторов и соответствие набора CDS исходной аннотации Bakta. Для каждой записи сопоставили координаты и направление цепи с GenBank. Подтвердили сохранение всех 4315 CDS.

По белковому FASTA сформировали соответствие между ID Bakta и аминокислотной последовательностью. Этот набор использовали при проверке компонентов DefenseFinder.

</div>

In [3]:
bakta_cds = bakta_features.loc[bakta_features.feature_type.eq("CDS")].copy()
primary = pd.read_csv(INPUTS["functional_table"], sep="\t")
assert primary["query"].is_unique and primary["query"].notna().all()
assert bakta_cds.locus_tag.is_unique and len(primary) == len(bakta_cds) == 4315
assert set(primary["query"]) == set(bakta_cds.locus_tag), "Bakta CDS IDs mismatch"

expected = bakta_cds.set_index("locus_tag").loc[primary["query"]]
for field in ("start", "end", "strand"):
    assert (primary[field].to_numpy() == expected[field].to_numpy()).all(), f"Field mismatch: {field}"

protein_records = list(SeqIO.parse(INPUTS["bakta_faa"], "fasta"))
proteins = {r.id: str(r.seq).upper().rstrip("*") for r in protein_records}
assert len(proteins) == len(protein_records) == len(primary)
assert set(proteins) == set(primary["query"])
print("Bakta CDS:", len(primary))
print("Hypothetical proteins:", primary.bakta_product.fillna("").str.lower().eq("hypothetical protein").sum())

Bakta CDS: 4315
Hypothetical proteins: 234


<div style="line-height: 1.8;">

## **4. Специализированные предсказания и привязка защитных белков**

В подготовленных таблицах PHASTEST и DefenseFinder проверили допустимость геномных интервалов. Учли пять профагоподобных областей, девять защитных систем и 23 белковых компонента. Эти объекты сохранили как дополнительные признаки без расширения основного набора CDS.

Привязку компонента DefenseFinder к Bakta принимали при однозначном совпадении координат, цепи и белковой последовательности. Такое соответствие подтвердили для всех 23 компонентов. Для областей PHASTEST определили пересечение с CDS; координатное перекрытие не рассматривали как доказательство фагового происхождения каждого белка.

Проверку ограничили сохранёнными таблицами и последовательностями. Полные исходные ответы веб-сервисов в проекте отсутствовали, поэтому происхождение всех промежуточных результатов независимо не восстановили.

</div>

In [4]:
ph = pd.read_csv(INPUTS["phastest_regions"], sep="\t")
df_sys = pd.read_csv(INPUTS["defense_systems"], sep="\t")
df_genes = pd.read_csv(INPUTS["defense_genes"], sep="\t")
defense_peptides = {r.id: str(r.seq).upper().rstrip("*") for r in SeqIO.parse(INPUTS["defense_faa"], "fasta")}

def check_intervals(df, label):
    st = pd.to_numeric(df["start_1based"], errors="raise")
    en = pd.to_numeric(df["end_1based"], errors="raise")
    assert (st.ge(1) & en.le(GENOME_LENGTH) & en.ge(st)).all(), f"Invalid coordinates: {label}"

for d, label in ((ph,"PHASTEST"),(df_sys,"DefenseFinder systems"),(df_genes,"DefenseFinder genes")):
    check_intervals(d, label)
assert len(ph) == 5 and len(df_sys) == 9 and len(df_genes) == 23, "Unexpected region or system counts"
assert set(df_genes.hit_id) == set(defense_peptides), "DefenseFinder protein IDs mismatch"

coord_to_ids = {}
for row in primary.itertuples(index=False):
    coord_to_ids.setdefault((row.start, row.end, row.strand), []).append(str(row.query))
peptide_to_ids = {}
for protein_id, aa in proteins.items():
    peptide_to_ids.setdefault(aa, []).append(protein_id)

def classify_defense_hit(row):
    strand = {"+":1, "-":-1}[row.strand]
    exact_ids = coord_to_ids.get((int(row.start_1based), int(row.end_1based), strand), [])
    aa = defense_peptides[str(row.hit_id)]
    exact_protein_ids = [x for x in exact_ids if proteins[x] == aa]
    if len(exact_protein_ids) == 1:
        return pd.Series({"bakta_query":exact_protein_ids[0], "mapping_status":"exact_coordinates_and_protein"})
    if exact_ids:
        return pd.Series({"bakta_query":pd.NA, "mapping_status":"exact_coordinate_but_protein_ambiguous_or_mismatch_REVIEW"})
    ids = peptide_to_ids.get(aa, [])
    if len(ids) == 1:
        return pd.Series({"bakta_query":pd.NA, "mapping_status":"unique_protein_but_coordinates_differ_REVIEW"})
    if len(ids) > 1:
        return pd.Series({"bakta_query":pd.NA, "mapping_status":"nonunique_protein_REVIEW"})
    return pd.Series({"bakta_query":pd.NA, "mapping_status":"not_matched_REVIEW"})

defense_map = pd.concat([df_genes, df_genes.apply(classify_defense_hit,axis=1)], axis=1)
defense_map.to_csv(TABLES / "defense_genes_bakta_mapping.tsv", sep="\t", index=False)

good_defense = defense_map.loc[defense_map.mapping_status.eq("exact_coordinates_and_protein")]
defense_grouped = good_defense.groupby("bakta_query").sys_id.apply(lambda x:";".join(sorted(set(x)))).to_dict()
primary["defensefinder_system_ids_exact"] = primary["query"].astype(str).map(defense_grouped)

def prophage_overlap(row):
    return ";".join(str(x.region) for x in ph.itertuples(index=False)
        if row.start <= x.end_1based and row.end >= x.start_1based) or None
primary["phastest_overlapping_region_ids"] = primary.apply(prophage_overlap, axis=1)

print("DefenseFinder exact Bakta links:",len(good_defense),"/",len(defense_map))
print("DefenseFinder unresolved:",len(defense_map) - len(good_defense))
print("PHASTEST regions:",len(ph))
primary.to_csv(TABLES / "bakta_CDS_functional_annotation.tsv", sep="\t", index=False)

DefenseFinder exact Bakta links: 23 / 23
DefenseFinder unresolved: 0
PHASTEST regions: 5


<div style="line-height: 1.8;">

## **5. Отдельные GFF3-треки для PHASTEST и DefenseFinder**

Предсказанные области записали в отдельные GFF3 с идентификатором хромосомы, согласованным с Bakta. Классификацию PHASTEST `intact` сохранили как вычислительную характеристику и не интерпретировали как подтверждение жизнеспособности фага.

Исходные GenBank и GFF3 Bakta включили в выпуск без изменений. Дополнительные функциональные сведения и связи с системами сохранили в TSV и специализированных треках; в основной GenBank эти дополнения не вносили.

</div>

In [5]:
def gff_escape(value):
    return quote(str(value), safe="-_.~")

def write_region_track(df, source, key, label, target, extras):
    with open(target, "w", encoding="utf-8") as fp:
        fp.write("##gff-version 3\n")
        fp.write(f"##sequence-region {SEQID} 1 {GENOME_LENGTH}\n")
        for row in df.to_dict("records"):
            attrs = [f"ID={gff_escape(source + '_' + str(row[key]))}",
                     f"Name={gff_escape(row[label])}"]
            for attr, val in extras(row).items():
                if pd.notna(val):
                    attrs.append(f"{gff_escape(attr)}={gff_escape(val)}")
            fp.write("\t".join([SEQID, source, "region", str(int(row["start_1based"])),
                       str(int(row["end_1based"])), ".", ".", ".", ";".join(attrs)]) + "\n")

write_region_track(ph,"PHASTEST","region","region",TRACKS/"phastest_regions.gff3",
                   lambda r:{"classification":r["PHASTEST_class"],"score":r["PHASTEST_score"],
                             "Note":"computational_prediction_not_proof_of_viability"})
write_region_track(df_sys,"DefenseFinder","sys_id","type",TRACKS/"defense_systems.gff3",
                   lambda r:{"subtype":r["subtype"],"Note":"computational_prediction_not_proof_of_activity"})
ph.to_csv(TABLES/"phastest_regions.tsv",sep="\t",index=False)
df_sys.to_csv(TABLES/"defense_systems.tsv",sep="\t",index=False)

shutil.copy2(INPUTS["bakta_gff3"], RELEASE/"ecoli_bakta_primary.gff3")
shutil.copy2(INPUTS["bakta_gbff"], RELEASE/"ecoli_bakta_primary.gbff")

<div style="line-height: 1.8;">

## **6. Валидация по референсу NCBI**

После подтверждения идентичности последовательностей сопоставили расположение CDS Bakta и NCBI. При сравнении учли цепь и все части записей, поскольку 20 референсных CDS имели составную структуру.

Точное соответствие установили для 3908 CDS. Без точного совпадения с NCBI осталось 407 CDS Bakta; без точного совпадения с Bakta — 410 CDS NCBI. Таблицы соответствий сохранили отдельно от основного каталога. Расхождение с референсом не приравнивали к доказанной ошибке предсказания.

</div>

In [6]:
ncbi_qc = {"reference_available":NCBI_GBK.is_file(), "reference_sequence_exact":None,
           "ncbi_cds":None,"bakta_cds":len(bakta_cds),"exact_cds_coordinate_matches":None}
if NCBI_GBK.is_file():
    reference = SeqIO.read(NCBI_GBK, "genbank")
    assert str(reference.seq).upper() == str(input_genome.seq).upper(), "NCBI sequence mismatch"
    ncbi_rows = []
    for f in reference.features:
        if f.type != "CDS" or f.location is None:
            continue
        ncbi_rows.append({"start":int(f.location.start)+1,"end":int(f.location.end),
                          "strand":f.location.strand,"gene":qual_first(f,"gene"),
                          "locus_tag":qual_first(f,"locus_tag"),
                          "location_genbank":str(f.location),"is_compound":len(f.location.parts)>1})
    ref_cds = pd.DataFrame(ncbi_rows)
    key = ["location_genbank"]
    bk_coords = set(map(tuple, bakta_cds[key].itertuples(index=False, name=None)))
    nc_coords = set(map(tuple, ref_cds[key].itertuples(index=False, name=None)))
    ref_cds["exact_in_bakta"] = [k in bk_coords for k in ref_cds[key].itertuples(index=False,name=None)]
    bakta_validation = bakta_cds[["locus_tag","start","end","strand",*key]].copy()
    bakta_validation["exact_in_ncbi"] = [k in nc_coords for k in bakta_validation[key].itertuples(index=False,name=None)]
    ref_cds.to_csv(QC/"NCBI_CDS_validation_only.tsv",sep="\t",index=False)
    bakta_validation.to_csv(QC/"Bakta_CDS_validation_only.tsv",sep="\t",index=False)
    ncbi_qc.update({"reference_sequence_exact":True,"ncbi_cds":len(ref_cds),
                    "ncbi_compound_cds":int(ref_cds.is_compound.sum()),
                    "comparison_basis":"full feature location including parts and strand",
                    "exact_cds_coordinate_matches":int(bakta_validation.exact_in_ncbi.sum()),
                    "bakta_without_exact_reference_match":int((~bakta_validation.exact_in_ncbi).sum()),
                    "ncbi_without_exact_bakta_match":int((~ref_cds.exact_in_bakta).sum())})
    print("NCBI validation:",ncbi_qc)
else:
    print("NCBI validation skipped: reference unavailable")

NCBI validation: {'reference_available': True, 'reference_sequence_exact': True, 'ncbi_cds': 4318, 'bakta_cds': 4315, 'exact_cds_coordinate_matches': 3908, 'ncbi_compound_cds': 20, 'comparison_basis': 'full feature location including parts and strand', 'bakta_without_exact_reference_match': 407, 'ncbi_without_exact_bakta_match': 410}


<div style="line-height: 1.8;">

## **7. RegulonDB как отдельный источник**

Наборы RegulonDB сохранили в локальном каталоге, отдельно от объединённой аннотации. В соответствии с принятым для проекта порядком работы с условиями использования содержимое TSV не включали в публичный репозиторий.

В состав выпуска вошёл манифест с именами файлов, отметками наличия и контрольными суммами. Регуляторные сведения в основной каталог CDS и GFF3 не переносили.

</div>

In [7]:
def sha256(path):
    h = hashlib.sha256()
    with open(path,"rb") as fh:
        for chunk in iter(lambda:fh.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

reg_dir = PROJECT_ROOT/"results/regulatory/regulondb"
reg_files = ["GeneProductAllIdentifiersSet.tsv","OperonSet.tsv","TUSet.tsv",
             "PromoterSet.tsv","TerminatorSet.tsv","RISet.tsv","EvidenceSet.tsv","RegulatorSet.tsv"]
manifest = pd.DataFrame([{"filename":f,"present":(reg_dir/f).is_file(),
                           "sha256":sha256(reg_dir/f) if (reg_dir/f).is_file() else None}
                          for f in reg_files])
manifest.to_csv(QC/"regulondb_manifest_local_only.tsv",sep="\t",index=False)
display(manifest)

,filename,present,sha256
0,GeneProductAllIdentifiersSet.tsv,True,d77e5d9d5a4b7f21d2e50b6ac86eb4c283cf18c7fd6fba...
1,OperonSet.tsv,True,49cdd547fb9b312d8fa07bf71fd943fd607bf1871fd65c...
2,TUSet.tsv,True,5d7eb8a014fa2c21ad7c2abe7c0482f353e6b9741b17c9...
3,PromoterSet.tsv,True,8d42457481cd99534ad7ce9ef12a32721e26345e5d8e2c...
4,TerminatorSet.tsv,True,1178585a0ab494d13268273f5770ea2a4c0560691b197d...
5,RISet.tsv,True,cfb47da9a7e0b15bd4bb54c68c870ce0b47d835a4d7a6f...
6,EvidenceSet.tsv,True,425564c18317a45a4b320b1a29ebcc7a9d5735b51c563b...
7,RegulatorSet.tsv,True,3505a4fb691a81a5a14f8da86687a36b51ce49132335be...


<div style="line-height: 1.8;">

## **8. Сводка и контрольные суммы выпуска**

В `qc/summary.json` зафиксировали численность геномных признаков, специализированных областей и систем, результаты сравнения с NCBI и ограничения анализа. Для входных файлов и результатов рассчитали контрольные суммы SHA-256; файл самого перечня контрольных сумм из расчёта исключили.

</div>

In [8]:
summary = {
    "release":"v1.0-bakta-primary",
    "primary_annotation":"Bakta independent FASTA-based prediction",
    "genome_length_bp":GENOME_LENGTH,
    "primary_bakta_cds":int(len(bakta_cds)),
    "primary_bakta_features_total":int(len(bakta_features)),
    "hypothetical_proteins_Bakta":int(primary.bakta_product.fillna("").str.lower().eq("hypothetical protein").sum()),
    "enriched_CDS_rows":int(len(primary)),
    "PHASTEST_predicted_regions":int(len(ph)),
    "DefenseFinder_predicted_systems":int(len(df_sys)),
    "DefenseFinder_components":int(len(defense_map)),
    "DefenseFinder_exact_matches_to_Bakta":int(len(good_defense)),
    "DefenseFinder_requires_review":int(len(defense_map)-len(good_defense)),
    "NCBI_validation_only":ncbi_qc,
    "limitations":["Bakta light DB used (not full)",
        "InterProScan only for 234 initially hypothetical Bakta proteins",
        "RegulonDB datasets excluded from integrated release pending license clearance",
        "CRISPR spacer annotation not completed",
        "Metabolic pathway reconstruction not performed",
        "PHASTEST/DefenseFinder systems are predictions, not biological activity proofs"]
}
(QC/"summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")

checksum_records=[]
checksum_inputs = dict(INPUTS)
if NCBI_GBK.is_file():
    checksum_inputs["ncbi_reference_qc"] = NCBI_GBK
for name,path in checksum_inputs.items():
    checksum_records.append({"kind":"input","name":name,"path":str(path.relative_to(PROJECT_ROOT)),"sha256":sha256(path)})
for path in sorted(RELEASE.rglob("*")):
    if path.is_file() and path.name != "checksums.tsv":
        checksum_records.append({"kind":"output","name":path.name,
                                 "path":str(path.relative_to(PROJECT_ROOT)),"sha256":sha256(path)})
pd.DataFrame(checksum_records).to_csv(QC/"checksums.tsv",sep="\t",index=False)
print(json.dumps(summary,ensure_ascii=False,indent=2))

{
  "release": "v1.0-bakta-primary",
  "primary_annotation": "Bakta independent FASTA-based prediction",
  "genome_length_bp": 4641652,
  "primary_bakta_cds": 4315,
  "primary_bakta_features_total": 9378,
  "hypothetical_proteins_Bakta": 234,
  "enriched_CDS_rows": 4315,
  "PHASTEST_predicted_regions": 5,
  "DefenseFinder_predicted_systems": 9,
  "DefenseFinder_components": 23,
  "DefenseFinder_exact_matches_to_Bakta": 23,
  "DefenseFinder_requires_review": 0,
  "NCBI_validation_only": {
    "reference_available": true,
    "reference_sequence_exact": true,
    "ncbi_cds": 4318,
    "bakta_cds": 4315,
    "exact_cds_coordinate_matches": 3908,
    "ncbi_compound_cds": 20,
    "comparison_basis": "full feature location including parts and strand",
    "bakta_without_exact_reference_match": 407,
    "ncbi_without_exact_bakta_match": 410
  },
  "limitations": [
    "Bakta light DB used (not full)",
    "InterProScan only for 234 initially hypothetical Bakta proteins",
    "RegulonDB datase

<div style="line-height: 1.8;">

## **Интерпретация и статус**

В результате сформировали выпуск с исходной структурной аннотацией Bakta, таблицей всех геномных признаков, функциональным каталогом 4315 CDS, отдельными треками PHASTEST и DefenseFinder и материалами контроля качества.

При интерпретации учли использование базы Bakta `light` и ограничение анализа InterProScan набором из 234 исходных гипотетических белков. Полную аннотацию CRISPR-спейсеров и реконструкцию метаболизма не выполняли. Полученные функциональные признаки и специализированные системы рассматривали как вычислительные результаты, без экспериментального подтверждения их активности.

</div>